# PyTorch AQI Sequence Forecasting

This notebook adds a PyTorch sequence-model workflow on top of the canonical Open-Meteo hourly table. It is designed to stay CPU-friendly while still comparing several recurrent models side by side.

The workflow is:

- build rolling 24-hour windows from the canonical hourly table
- predict the next hour of `european_aqi`
- compare compact LSTM and GRU baselines
- inspect learning curves and a short forecast window

In [ ]:
from __future__ import annotations

import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from open_meteo import DEFAULT_DATA_DIR, build_hourly_table

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device('cpu')
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print({'device': str(device), 'torch_version': torch.__version__})

## Load the canonical hourly table

The notebook reuses the same aligned hourly table as the CLI and sklearn notebooks so all comparisons stay on the same data foundation.

In [ ]:
table = build_hourly_table(DEFAULT_DATA_DIR)
column_arrays = {name: np.asarray(values, dtype=float) for name, values in table.columns.items()}
timestamps = np.asarray(table.timestamps)

target_name = 'european_air_quality_index__european_aqi'
feature_names = [
    'european_air_quality_index__european_aqi',
    'hourly_air_quality_vars__pm2_5',
    'hourly_air_quality_vars__pm10',
    'hourly_air_quality_vars__nitrogen_dioxide',
    'hourly_air_quality_vars__ozone',
    'hourly_air_quality_vars__sulphur_dioxide',
    'hourly_air_quality_vars__carbon_monoxide',
    'misc__temperature_2m',
    'misc__relative_humidity_2m',
    'misc__pressure_msl',
    'misc__precipitation',
    'wind__wind_speed_10m',
    'cloud_cover__cloud_cover',
    'solar_radiation_vars__shortwave_radiation',
    'additional_vars_and_options__is_day',
    'air_quality_us_index__us_aqi',
]
missing = [name for name in feature_names if name not in column_arrays]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

feature_matrix = np.column_stack([column_arrays[name] for name in feature_names]).astype(np.float32)
target_values = column_arrays[target_name].astype(np.float32)

print(f'Rows: {len(timestamps):,}')
print(f'Feature columns: {len(feature_names)}')
print(f'Start: {timestamps[0]}')
print(f'End:   {timestamps[-1]}')

## Build rolling sequence windows

Each sample uses the previous 24 hourly rows to predict the next hour of `european_aqi`. To keep the notebook CPU-friendly, the benchmark uses only the most recent subset of valid windows.

In [ ]:
sequence_length = 24
max_windows = 18000

windows = []
targets = []
window_times = []

for end_index in range(sequence_length, len(timestamps)):
    window = feature_matrix[end_index - sequence_length:end_index]
    target = target_values[end_index]
    if np.isfinite(window).all() and np.isfinite(target):
        windows.append(window)
        targets.append(target)
        window_times.append(timestamps[end_index])

X = np.asarray(windows, dtype=np.float32)
y = np.asarray(targets, dtype=np.float32)
time_index = np.asarray(window_times)

if len(y) > max_windows:
    X = X[-max_windows:]
    y = y[-max_windows:]
    time_index = time_index[-max_windows:]

print(f'Windowed samples: {len(y):,}')
print(f'Window shape: {X.shape[1:]}')
print(f'Subset start: {time_index[0]}')
print(f'Subset end:   {time_index[-1]}')

In [ ]:
def chronological_split(X: np.ndarray, y: np.ndarray, time_index: np.ndarray, train_ratio: float = 0.7, valid_ratio: float = 0.15):
    total = len(y)
    train_end = int(total * train_ratio)
    valid_end = int(total * (train_ratio + valid_ratio))
    return {
        'X_train': X[:train_end],
        'y_train': y[:train_end],
        't_train': time_index[:train_end],
        'X_valid': X[train_end:valid_end],
        'y_valid': y[train_end:valid_end],
        't_valid': time_index[train_end:valid_end],
        'X_test': X[valid_end:],
        'y_test': y[valid_end:],
        't_test': time_index[valid_end:],
    }

split = chronological_split(X, y, time_index)

train_flat = split['X_train'].reshape(-1, split['X_train'].shape[-1])
x_mean = train_flat.mean(axis=0)
x_std = train_flat.std(axis=0) + 1e-6
y_mean = split['y_train'].mean()
y_std = split['y_train'].std() + 1e-6

def normalize_features(values: np.ndarray) -> np.ndarray:
    return ((values - x_mean) / x_std).astype(np.float32)

def normalize_targets(values: np.ndarray) -> np.ndarray:
    return ((values - y_mean) / y_std).astype(np.float32)

def to_tensor_dataset(X_values: np.ndarray, y_values: np.ndarray) -> TensorDataset:
    return TensorDataset(
        torch.from_numpy(normalize_features(X_values)),
        torch.from_numpy(normalize_targets(y_values)),
    )

train_dataset = to_tensor_dataset(split['X_train'], split['y_train'])
valid_dataset = to_tensor_dataset(split['X_valid'], split['y_valid'])
test_dataset = to_tensor_dataset(split['X_test'], split['y_test'])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

for name in ['train', 'valid', 'test']:
    print(name, split[f'X_{name}'].shape, split[f't_{name}'][0], split[f't_{name}'][-1])

## Define compact PyTorch sequence models

The benchmark keeps models intentionally small so the notebook stays practical on CPU while still showing meaningful differences between recurrent architectures.

In [ ]:
class SequenceRegressor(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, rnn_type: str, num_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        rnn_cls = nn.LSTM if rnn_type == 'lstm' else nn.GRU
        self.encoder = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        head_hidden = max(16, hidden_size // 2)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, head_hidden),
            nn.ReLU(),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, hidden = self.encoder(x)
        if isinstance(hidden, tuple):
            hidden_state = hidden[0][-1]
        else:
            hidden_state = hidden[-1]
        return self.head(hidden_state).squeeze(-1)

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(mean_squared_error(y_true, y_pred, squared=False)),
        'r2': float(r2_score(y_true, y_pred)),
    }

def denormalize_targets(values: np.ndarray) -> np.ndarray:
    return values * y_std + y_mean

def evaluate_model(model: nn.Module, loader: DataLoader) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    loss_fn = nn.MSELoss()
    losses = []
    preds = []
    targets = []
    with torch.no_grad():
        for features, target in loader:
            features = features.to(device)
            target = target.to(device)
            output = model(features)
            loss = loss_fn(output, target)
            losses.append(loss.item())
            preds.append(output.cpu().numpy())
            targets.append(target.cpu().numpy())
    pred_array = denormalize_targets(np.concatenate(preds))
    target_array = denormalize_targets(np.concatenate(targets))
    return float(np.mean(losses)), pred_array, target_array

def train_model(model: nn.Module, train_loader: DataLoader, valid_loader: DataLoader, epochs: int = 8, lr: float = 1e-3, weight_decay: float = 1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    history = []
    best_valid_loss = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for features, target in train_loader:
            features = features.to(device)
            target = target.to(device)
            optimizer.zero_grad()
            output = model(features)
            loss = loss_fn(output, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))
        valid_loss, _, _ = evaluate_model(model, valid_loader)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'valid_loss': valid_loss})

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return history

## Compare PyTorch sequence baselines

The benchmark below keeps the model set small enough for CPU use while still comparing distinct recurrent architectures.

In [ ]:
benchmark_specs = {
    'lstm_small': {'rnn_type': 'lstm', 'hidden_size': 32, 'num_layers': 1, 'dropout': 0.0},
    'gru_small': {'rnn_type': 'gru', 'hidden_size': 32, 'num_layers': 1, 'dropout': 0.0},
    'gru_stacked': {'rnn_type': 'gru', 'hidden_size': 48, 'num_layers': 2, 'dropout': 0.1},
}

benchmark_rows = []
histories = {}
test_predictions = {}

for model_name, spec in benchmark_specs.items():
    model = SequenceRegressor(
        input_size=len(feature_names),
        hidden_size=spec['hidden_size'],
        rnn_type=spec['rnn_type'],
        num_layers=spec['num_layers'],
        dropout=spec['dropout'],
    ).to(device)
    history = train_model(model, train_loader, valid_loader, epochs=8, lr=1e-3, weight_decay=1e-4)
    valid_loss, valid_pred, valid_true = evaluate_model(model, valid_loader)
    test_loss, test_pred, test_true = evaluate_model(model, test_loader)
    valid_scores = regression_metrics(valid_true, valid_pred)
    test_scores = regression_metrics(test_true, test_pred)
    benchmark_rows.append({
        'model': model_name,
        'valid_mae': valid_scores['mae'],
        'valid_rmse': valid_scores['rmse'],
        'test_mae': test_scores['mae'],
        'test_rmse': test_scores['rmse'],
        'test_r2': test_scores['r2'],
    })
    histories[model_name] = history
    test_predictions[model_name] = test_pred

for row in sorted(benchmark_rows, key=lambda item: item['test_rmse']):
    print(row)

In [ ]:
best_model_name = min(benchmark_rows, key=lambda item: item['test_rmse'])['model']
plot_count = min(24 * 7, len(split['y_test']))

fig, ax = plt.subplots()
ax.plot(split['t_test'][:plot_count], split['y_test'][:plot_count], label='actual', linewidth=2)
ax.plot(split['t_test'][:plot_count], test_predictions[best_model_name][:plot_count], label=best_model_name, linewidth=1.8)
ax.set_title('Next-hour European AQI forecast on the first week of the test split')
ax.set_xlabel('Timestamp')
ax.set_ylabel('European AQI')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()

fig, axes = plt.subplots(1, len(histories), figsize=(5 * len(histories), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (model_name, history) in zip(axes, histories.items()):
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    valid_loss = [row['valid_loss'] for row in history]
    ax.plot(epochs, train_loss, label='train')
    ax.plot(epochs, valid_loss, label='valid')
    ax.set_title(model_name)
    ax.set_xlabel('Epoch')
    ax.legend()
axes[0].set_ylabel('MSE loss on normalized target')
plt.tight_layout()

## Suggested follow-up experiments

- widen the sequence window from 24 hours to 48 or 72 hours
- add multistep targets such as 6-hour or 24-hour AQI forecasting
- compare these PyTorch sequence models with the sklearn tabular baselines
- enable MPS later if you want a longer training budget on Apple Silicon